# 02B Portable Feature Ablation

This notebook runs controlled portable-model ablations on dataset `P` so you can compare which feature profile actually helps transfer.

In [ ]:
from pathlib import Path
import pandas as pd
import subprocess
import sys

ROOT = Path('/Users/k.e.oshada/Documents/OptiWMS/Ai miroservices/modeling')
REPORTS_DIR = ROOT / 'outputs' / 'reports'
SCRIPT = ROOT / 'scripts' / 'train_and_save_models.py'

DATASET = 'P'
MODELS = ['XGBOOST']
FEATURE_PROFILES = [
    'lags_only',
    'lags_roll',
    'lags_roll_seasonal',
    'lags_roll_seasonal_category',
    'full',
]
TAG_PREFIX = 'portable_ablation'
HORIZONS = '1,2,3,4,5,6,7,8,9,10,11,12'


## Run Training

Run this cell once. It trains one tag per feature profile.

In [ ]:
for profile in FEATURE_PROFILES:
    tag = f'{TAG_PREFIX}_{profile}'
    cmd = [
        sys.executable,
        str(SCRIPT),
        '--datasets', DATASET,
        '--skip-classical',
        '--boosting-models', *MODELS,
        '--feature-profile', profile,
        '--horizons', HORIZONS,
        '--tag', tag,
    ]
    print('RUNNING', ' '.join(cmd))
    subprocess.run(cmd, check=True)


## Compare Leaderboards

In [ ]:
rows = []
for profile in FEATURE_PROFILES:
    tag = f'{TAG_PREFIX}_{profile}'
    path = REPORTS_DIR / f'{tag}_leaderboard.csv'
    if not path.exists():
        continue
    df = pd.read_csv(path)
    df['feature_profile'] = profile
    rows.append(df)

ablation = pd.concat(rows, ignore_index=True)
ablation[['feature_profile', 'dataset', 'model', 'WAPE', 'RMSE', 'Bias', 'MASE_mean']].sort_values(['WAPE', 'RMSE'])


## Inspect One Profile

In [ ]:
PROFILE_TO_VIEW = 'full'
tag = f'{TAG_PREFIX}_{PROFILE_TO_VIEW}'
pd.read_csv(REPORTS_DIR / f'{tag}_p_metrics.csv').head(30)
